# BizdenBize – Smoke & Fire Detector (YOLOv8s)
Run on Google Colab with a **T4 GPU** (Runtime > Change runtime type).
Output: `best.onnx` → copy to `verifier/models/best.onnx`.

In [ ]:
!pip -q install ultralytics==8.3.40

## 1. Get D-Fire
YOLO format: `0=smoke`, `1=fire`; an empty `.txt` label = hard negative (sunset/fog/cloud).
Download from the D-Fire GitHub repo (gaiasd/DFireDataset) into Drive. Expected layout: `DFIRE_ROOT/{train,test}/{images,labels}`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DFIRE_ROOT = '/content/drive/MyDrive/DFireDataset'   # <-- edit
OUT = '/content/dfire_subset'

## 2. Build a balanced ~3,000-image subset
40% smoke, 25% fire, 35% hard negatives. Optionally add your own Antalya / Kalimantan frames (+ YOLO labels) to `OUT/images/train` and `OUT/labels/train` after this cell.

In [ ]:
import random, shutil, pathlib
random.seed(42)

def classes_in(label_path):
    p = pathlib.Path(label_path)
    txt = p.read_text().strip() if p.exists() else ''
    return {int(l.split()[0]) for l in txt.splitlines()} if txt else set()

def bucket(split):
    img_dir = pathlib.Path(DFIRE_ROOT, split, 'images')
    lbl_dir = pathlib.Path(DFIRE_ROOT, split, 'labels')
    smoke, fire, neg = [], [], []
    for img in img_dir.glob('*.jpg'):
        c = classes_in(lbl_dir / (img.stem + '.txt'))
        (fire if 1 in c else smoke if 0 in c else neg).append(img)
    return smoke, fire, neg

def copy(items, split):
    for img in items:
        lbl = img.parent.parent / 'labels' / (img.stem + '.txt')
        for src, sub in ((img, 'images'), (lbl, 'labels')):
            dst = pathlib.Path(OUT, sub, split); dst.mkdir(parents=True, exist_ok=True)
            if src.exists(): shutil.copy(src, dst / src.name)
            else: (dst / src.name).write_text('')

TOTAL = 3000
smoke, fire, neg = bucket('train')
print('available  smoke', len(smoke), 'fire', len(fire), 'neg', len(neg))
pick = (random.sample(smoke, min(len(smoke), int(TOTAL*.40))) +
        random.sample(fire,  min(len(fire),  int(TOTAL*.25))) +
        random.sample(neg,   min(len(neg),   int(TOTAL*.35))))
random.shuffle(pick)
cut = int(len(pick) * .85)
copy(pick[:cut], 'train'); copy(pick[cut:], 'val')
print('train', cut, 'val', len(pick) - cut)

open('/content/dfire.yaml', 'w').write(f'''path: {OUT}
train: images/train
val: images/val
names: {{0: smoke, 1: fire}}
''')

## 3. Train (~25 min on T4)

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(data='/content/dfire.yaml', epochs=25, imgsz=640, batch=32, patience=8)

## 4. Evaluate
Check **smoke** mAP50 separately, and the false-positive rate on hard negatives — this is the number the jury cares about.

In [ ]:
best = YOLO('runs/detect/train/weights/best.pt')
m = best.val(data='/content/dfire.yaml')
print('mAP50 per class (smoke, fire):', m.box.ap50)

neg_val = [str(p) for p in pathlib.Path(OUT, 'images', 'val').glob('*')
           if not pathlib.Path(OUT, 'labels', 'val', p.stem + '.txt').read_text().strip()]
fp = sum(len(r.boxes) > 0 for r in best.predict(neg_val, conf=0.35, verbose=False, stream=True))
print(f'False positives on sunsets/fog/clouds: {fp}/{len(neg_val)} = {fp/max(1,len(neg_val)):.1%}')

## 5. Export ONNX and download

In [ ]:
path = best.export(format='onnx', imgsz=640, simplify=True)
from google.colab import files
files.download(path)